# One-dimensional log-P-spline PSD model

For a stationary time series, the power spectral density depends only on frequency,

$$
S(f).
$$

`log_psplines` models the PSD using a smooth spline representation of the
**log spectrum**,

$$
\eta(f) \equiv \log S(f).
$$

The PSD is then recovered through

$$
S(f) = \exp\{\eta(f)\},
$$

which guarantees that

$$
S(f) > 0.
$$

## Frequency-domain likelihood

Let

$$
d(f_k)
$$

denote the Fourier coefficient of the time series at frequency $f_k$.

Under the Whittle approximation, Fourier coefficients at distinct frequencies
are treated as approximately independent complex Gaussian variables,

$$
d(f_k)
\sim
\mathcal{CN}
\left(
0,
T\,S(f_k)
\right),
$$

where $T$ is the observation duration.

The corresponding periodogram ordinate is

$$
I(f_k)
=
|d(f_k)|^2.
$$

Up to constants independent of the PSD, the Whittle log-likelihood is

$$
\boxed{
\log \mathcal{L}
=
-
\sum_k
\log S(f_k)
-
\frac{1}{T}
\sum_k
\frac{I(f_k)}{S(f_k)}
}
$$

Thus, the periodogram provides a noisy observation of the underlying spectral
density, while the spline model regularises the spectrum across frequency.

## Log-P-spline model

Let

$$
\mathbf{b}(f)
=
\begin{bmatrix}
B_1(f) & \cdots & B_K(f)
\end{bmatrix}
$$

denote a B-spline basis evaluated at frequency $f$.

The log spectrum is represented as

$$
\boxed{
\log S(f)
=
\mathbf{b}(f)\mathbf{w}
}
$$

where

$$
\mathbf{w}
=
\begin{bmatrix}
w_1 & \cdots & w_K
\end{bmatrix}^{\mathsf{T}}
$$

contains the spline coefficients.

Equivalently,

$$
\boxed{
\log S(f)
=
\sum_{k=1}^{K}
w_k B_k(f)
}
$$

The PSD is therefore

$$
S(f)
=
\exp
\left[
\sum_{k=1}^{K}
w_k B_k(f)
\right].
$$

The B-spline basis controls the flexibility of the representation, while the
P-spline prior prevents the coefficients from varying too rapidly.

## Smoothness prior

Smoothness is imposed through a quadratic roughness penalty.

Let

$$
\mathbf{Q}
$$

denote the spline penalty matrix. The spline coefficients are assigned the
Gaussian prior

$$
\boxed{
\mathbf{w}\mid\phi
\sim
\mathcal{N}
\left(
\mathbf{0},
(\phi\mathbf{Q})^{-1}
\right)
}
$$

or, equivalently,

$$
p(\mathbf{w}\mid\phi)
\propto
\exp
\left[
-\frac{\phi}{2}
\mathbf{w}^{\mathsf{T}}
\mathbf{Q}
\mathbf{w}
\right].
$$

The quantity

$$
\mathbf{w}^{\mathsf{T}}
\mathbf{Q}
\mathbf{w}
$$

measures the roughness of the fitted log spectrum.

The smoothing precision

$$
\phi > 0
$$

controls how strongly this roughness is penalised:

* large $\phi$ favours a smoother spectrum,
* small $\phi$ allows greater variation between neighbouring spline
  coefficients.

## Hierarchical smoothing prior

Rather than fixing the amount of smoothing, the smoothing precision is inferred
from the data.

The hierarchical prior is

$$
\mathbf{w}\mid\phi
\sim
\mathcal{N}
\left(
\mathbf{0},
(\phi\mathbf{Q})^{-1}
\right),
$$

$$
\phi\mid\delta
\sim
\operatorname{Gamma}
\left(
\alpha_\phi,
\beta_\phi\delta
\right),
$$

and

$$
\delta
\sim
\operatorname{Gamma}
\left(
\alpha_\delta,
\beta_\delta
\right).
$$

This allows the data to determine how strongly the spline should be smoothed,
rather than requiring a fixed regularisation strength.

## Posterior

The posterior distribution is

$$
p(\mathbf{w},\phi,\delta\mid d)
\propto
\mathcal{L}(d\mid\mathbf{w})\,
p(\mathbf{w}\mid\phi)\,
p(\phi\mid\delta)\,
p(\delta).
$$

Since

$$
S(f)
=
\exp\{\mathbf{b}(f)\mathbf{w}\},
$$

each posterior sample of the spline coefficients defines a complete PSD curve.

Posterior draws of

$$
S(f)
$$

can therefore be used to compute point estimates, credible intervals, or other
derived spectral quantities.

## Interpretation

The model separates PSD estimation into two components:

1. the Whittle likelihood describes the noisy frequency-domain observations,
2. the P-spline prior enforces smoothness across frequency.

The number and placement of B-spline basis functions determine how flexible the
representation can be, while the smoothing precision $\phi$ determines how much
of that flexibility is actually used.

This provides a flexible Bayesian PSD estimator that can represent broad
spectral structure while regularising the noisy periodogram.


## Example

Here we fit a stationary PSD to a simulated univariate AR(4) time series. 




![](../_static/demo.png)

In [ ]:
! pip install LogPSplinePSD

In [ ]:
from log_psplines import PipelineConfig, fit
from log_psplines.example_datasets import VARMAData

data = VARMAData.ar(
    order=4,
    n_samples=8192,
    fs=64.0,
    seed=7,
)

result = fit(
    data.ts,
    PipelineConfig(
        n_knots=16,
        knot_kwargs={"method": "density"},
        vi_steps=200,
        n_warmup=100,
        n_samples=200,
        rng_key=7,
        true_psd=data.get_true_psd(),
    ),
)

|00:00| LogPSpline | INFO | VAR sanity check passed: stationary AR dynamics (companion spectral radius=0.880945) with finite samples. Empirical stationarity passed (max_mean_shift_z=0.013, max_var_ratio=1.005, covariance_rel_drift=0.005).
|00:00| LogPSpline | INFO | Wishart averaging (blocks=1): n=8192 -> Lb=8192, N=4096, p=1
|00:00| LogPSpline | INFO | Standardized data: scale ~2.44e+00
|00:00| LogPSpline | INFO | Inferred sampler type: multivar_blocked_nuts
|00:00| LogPSpline | INFO | Spline model: SpectralComponents(channels=1, knots=16, degree=3, penaltyOrder=2, N=4096, basis_shapes=[(4096, 18)])


sample: 100%|██████████| 300/300 [00:02<00:00, 119.01it/s, 511 steps of size 1.77e-02. acc. prob=0.95]
